In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

In [30]:
tor = pd.concat([pd.read_csv(f"../merged/merged_tornados_{i}.csv.gz") for i in range(2023, 2014, -1)], axis=0)

In [31]:
# Deletes the column named 'Unnamed: 0' from the DataFrame tor
del tor['Unnamed: 0']
#Converts the values in the 'DATE' column of the DataFrame tor to datetime objects
tor['DATE'] = pd.to_datetime(tor['DATE'])
# Extracts the month component from the datetime values in the 'DATE' column and assigns it to a new column named 'Month' in the DataFrame tor
tor['Month'] = tor['DATE'].dt.month

In [32]:
tor['y'] = tor['power_outage'].astype(int)

In [33]:
tor.columns

Index(['DATE', 'LAT_mean', 'LON_mean', 'AVGDV_max', 'LLDV_max', 'MXDV_max',
       'MXDV_HEIGHT_max', 'DEPTH_max', 'MAX_SHEAR_max', 'MAX_SHEAR_HEIGHT_max',
       'county', 'state', 'Event Month', 'power_outage', 'Month', 'y'],
      dtype='object')

In [34]:
from sklearn.model_selection import train_test_split

In [35]:
tor_train, tor_test = train_test_split(tor.copy(),
                                              shuffle=True,
                                              random_state=123,
                                              test_size=.2,
                                              stratify=tor.y.values)

In [36]:
tor_tt, tor_val = train_test_split(tor_train.copy(),
                                              shuffle=True,
                                              random_state=123,
                                              test_size=.2,
                                              stratify=tor_train.y.values)

In [37]:
outage = tor_tt[tor_tt['power_outage']==True]
no_outage = tor_tt[tor_tt['power_outage']==False]
no_outage= no_outage.sample(n=len(outage), random_state=101)
tor_tt_bal = pd.concat([outage,no_outage],axis=0)


In [38]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score


In [39]:
all_features= ['LAT_mean', 'LON_mean', 'AVGDV_max', 'LLDV_max', 'MXDV_max',
       'MXDV_HEIGHT_max', 'DEPTH_max', 'MAX_SHEAR_max', 'MAX_SHEAR_HEIGHT_max','Month']

In [40]:
## set the number of CV folds
n_splits = 5

## Make the kfold object
kfold = StratifiedKFold(n_splits, 
                        random_state=216, 
                        shuffle=True)

In [41]:
from sklearn.model_selection import GridSearchCV

In [42]:
max_depths = range(1, 11)
n_trees = [100, 500]

In [43]:
grid_cv = GridSearchCV(RandomForestClassifier(), # first put the model object here
                          param_grid = {'max_depth':max_depths, # place the grid values for max_depth and
                                        'n_estimators':n_trees}, # and n_estimators here
                          scoring = 'accuracy', # put the metric we are trying to optimize here as a string, "accuracy"
                          cv = 5) # put the number of cv splits here

## you fit it just like a model
grid_cv.fit(tor_tt_bal[all_features], tor_tt_bal.y)

GridSearchCV(cv=5, estimator=RandomForestClassifier(),
             param_grid={'max_depth': range(1, 11), 'n_estimators': [100, 500]},
             scoring='accuracy')

In [44]:
grid_cv.best_params_

{'max_depth': 10, 'n_estimators': 100}

In [45]:
grid_cv.best_score_

0.7490627160029598

In [46]:
grid_cv.best_estimator_

RandomForestClassifier(max_depth=10)

In [47]:
grid_cv.best_estimator_.predict(tor_val[all_features])

array([0, 0, 0, ..., 0, 0, 0])

In [48]:
grid_cv.cv_results_

{'mean_fit_time': array([ 0.50624876,  2.25718741,  0.86900001,  3.8003593 ,  0.9562016 ,
         5.09880095,  1.27502818,  6.73752418,  2.12328401, 10.48386736,
         2.15890236, 10.16029911,  2.24009581, 12.81962738,  2.93918648,
        14.10543094,  3.07611866, 18.77677665,  3.21952496, 19.9091557 ]),
 'std_fit_time': array([0.08616381, 0.09876334, 0.01433374, 0.22903334, 0.03670716,
        0.23258325, 0.07954732, 0.14247585, 0.19669396, 0.80324691,
        0.08959167, 0.4615648 , 0.09083104, 1.13230451, 0.17501664,
        1.07554181, 0.18039861, 2.07278852, 0.16823938, 2.92326326]),
 'mean_score_time': array([0.02056737, 0.0943418 , 0.02579021, 0.11644697, 0.02497711,
        0.15403104, 0.02855306, 0.14888535, 0.04998107, 0.20128322,
        0.0389668 , 0.17586899, 0.03935723, 0.23013787, 0.0474925 ,
        0.22769713, 0.04972868, 0.29733038, 0.05044737, 0.32215347]),
 'std_score_time': array([0.00326768, 0.00830316, 0.00109731, 0.01633695, 0.00192148,
        0.05606761, 

In [49]:
pd.DataFrame({'feature_importance_score':grid_cv.best_estimator_.feature_importances_},
                 index=all_features).sort_values('feature_importance_score',
                                                ascending=False)

,feature_importance_score
LON_mean,0.258966
LAT_mean,0.220555
Month,0.213166
MXDV_max,0.066353
DEPTH_max,0.049964
MAX_SHEAR_max,0.045809
LLDV_max,0.041825
MAX_SHEAR_HEIGHT_max,0.040949
AVGDV_max,0.036104
MXDV_HEIGHT_max,0.026310


We will do bagging.

In [50]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate

In [51]:
pipe = Pipeline([('scale', StandardScaler()),('knn',KNeighborsClassifier())])
bag_pipe = BaggingClassifier(pipe, bootstrap = True, max_samples = 0.90)
bag_cv = GridSearchCV(bag_pipe, 
                          param_grid = {'estimator__knn__n_neighbors':[1,2,3], 
                                        'n_estimators':np.arange(1,100,10)}, 
                          scoring = 'accuracy', 
                          cv = 5)
bag_cv.fit(tor_tt_bal[all_features], tor_tt_bal.y)

GridSearchCV(cv=5,
             estimator=BaggingClassifier(estimator=Pipeline(steps=[('scale',
                                                                    StandardScaler()),
                                                                   ('knn',
                                                                    KNeighborsClassifier())]),
                                         max_samples=0.9),
             param_grid={'estimator__knn__n_neighbors': [1, 2, 3],
                         'n_estimators': array([ 1, 11, 21, 31, 41, 51, 61, 71, 81, 91])},
             scoring='accuracy')

In [52]:
print(f"The best mean cv accuracy of {bag_cv.best_score_:.3f} was achieved using k = {bag_cv.best_estimator_.estimator['knn'].n_neighbors} and {bag_cv.best_estimator_.n_estimators} estimators")

The best mean cv accuracy of 0.674 was achieved using k = 3 and 91 estimators


In [53]:
single_pipe = Pipeline([('scale', StandardScaler()),('knn',KNeighborsClassifier(n_neighbors=3))])
single_cv = cross_validate(single_pipe, tor_tt_bal[all_features], tor_tt_bal.y, cv = 5, scoring = 'accuracy')

In [54]:
print(f"The mean cv accuracy of a single kNN model with k=3 is {single_cv['test_score'].mean():.3f}")

The mean cv accuracy of a single kNN model with k=3 is 0.665


In [55]:
model = grid_cv.best_estimator_

model.fit(tor_tt_bal[all_features], tor_tt_bal.y)

RandomForestClassifier(max_depth=10)

In [59]:
acc_b= accuracy_score(model.predict(tor_val[all_features]), tor_val.y)
prec_b= precision_score(model.predict(tor_val[all_features]), tor_val.y)
rec_b= recall_score(model.predict(tor_val[all_features]), tor_val.y)

In [60]:
print('Accuracy is', acc_b)
print('Precision is', prec_b)
print('Recall is', rec_b)

Accuracy is 0.7436124142908698
Precision is 0.7517810273715786
Recall is 0.12892232510288065
